# Mejoras para california-e2e


Proyecto muy probable para examen porque el dataset ya está en el repo.

Mejoras típicas:

- pipeline completo sin leakage
- ratios de habitaciones/personas
- log transform en variables sesgadas
- clustering geográfico
- búsqueda de hiperparámetros
- evaluación con RMSE/MAE/R2


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
DATA_PATH = Path("../proyects/california-e2e/data/housing.csv")

df = pd.read_csv(DATA_PATH)

# Estratificación por income_cat, igual que el proyecto original.
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)

train_set, test_set = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["income_cat"],
)

for split in (train_set, test_set):
    split.drop(columns=["income_cat"], inplace=True)

X_train = train_set.drop(columns=["median_house_value"])
y_train = train_set["median_house_value"]
X_test = test_set.drop(columns=["median_house_value"])
y_test = test_set["median_house_value"]


In [ ]:
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=42):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, n_init=10, random_state=self.random_state)
        self.kmeans_.fit(X)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)


def ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_names(transformer, feature_names_in):
    return ["ratio"]

ratio_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("ratio", FunctionTransformer(ratio, feature_names_out=ratio_names)),
    ("scaler", StandardScaler()),
])

log_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("bedrooms_ratio", ratio_pipeline, ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline, ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline, ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("geo", ClusterSimilarity(random_state=RANDOM_STATE), ["latitude", "longitude"]),
    ("cat", cat_pipeline, ["ocean_proximity"]),
], remainder=Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]))


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_grid = {
    "preprocessor__geo__n_clusters": [5, 10, 15],
    "preprocessor__geo__gamma": [0.1, 1.0, 10.0],
    "regressor__n_estimators": [100, 200],
    "regressor__max_depth": [None, 16, 24],
    "regressor__min_samples_leaf": [1, 3, 5],
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)
print("Mejor RMSE CV:", -search.best_score_)


In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.0f}")
print(f"MAE:  {mae:.0f}")
print(f"R2:   {r2:.4f}")


Explicación corta para el examen:

“Uso un pipeline para evitar leakage. Creo ratios porque en vivienda importan más las proporciones que los totales brutos. Aplico log a variables sesgadas y añado similitud geográfica con clusters. Ajusto hiperparámetros con validación cruzada y evalúo una sola vez en test.”
